# Data analysis
## Import libraries and load dataset

In [ ]:
import pandas as pd
from pandas import DataFrame
import os
import re
from collections import Counter
import matplotlib.pyplot as plt
import numpy as np
import emoji
import spacy
import it_core_news_sm
from multipride.preprocessing import clean_text

# Check the cwd
os.getcwd()

In [ ]:
# Import data from CSV
data = pd.read_csv("../data/raw/train_it.csv")
data

## Basic stats

Check how many tweets are present in the data set, the distribution of the two classes and other general info.

In [ ]:
# some basic stats
print(len(data))
print(data["label"].value_counts())
print(data.info())

The classes are pretty unbalanced: 807 tweets don't contain reclaimed slurs while 207 does.

Now we compute the average lenght of the tweets and the bios, to see if they'll fit the model constrains later.

In [ ]:
# Print average length of texts in tokens
data["text_length"] = data["text"].apply(lambda x: len(x.split()))
print(data["text_length"].mean())

# Print average length of biographies in tokens (if available, otherwise do not count them)
data["bio_length"] = data["bio"].apply(
    lambda x: len(x.split()) if pd.notnull(x) else None
)
print(data["bio_length"].mean())

## Lexical and Semantic Analysis

### Tokenization and Cleaning

Check for user mentions and url to see if they are useful to keep or not.

In [ ]:
# Check for user mentions
data["user_mention"] = data["text"].apply(lambda x: re.findall(r"\s@\w+", x))

data[data["user_mention"].apply(lambda x: len(x) > 0)]

In [ ]:
# Check for urls
data["urls"] = data["text"].apply(lambda x: re.findall(r"url", x))
data[data["urls"].apply(lambda x: len(x) > 0)]

In [ ]:
# Check for hashtags
data["hashtags"] = data["text"].apply(lambda x: re.findall(r"#\w+", x))
data[data["hashtags"].apply(lambda x: len(x) > 0)]

In [ ]:
# Check for emojis

data["emojis"] = data["text"].apply(lambda x: emoji.distinct_emoji_list(x))

data[data["emojis"].apply(lambda x: len(x) > 0)]

Urls have been replaced with the string "url". Since all user mentions are hidden for privacy, we can remove them without losing meaning.

In [ ]:
# Drop useless columns
data = data.drop(columns=["text_length", "bio_length", "user_mention", "urls"])

We use the next function to remove user mentions, extra space, and useless punctuation. We keep the Upper-case for the file that will be used to train the model.

In [ ]:
data["text_clean"] = data["text"].apply(lambda x: clean_text(x))

In [ ]:
data

In [ ]:
# Save clean data to csv
clean_data = data
clean_data["text"] = clean_data["text_clean"]

In [ ]:
clean_data = clean_data.drop(columns=["text_clean", "hashtags", "emojis"])

In [ ]:
clean_data.to_csv("../data/processed/clean_data.csv")

We need to transform all characters in lower-case for further analysis

In [ ]:
data["text_clean"] = data["text"].apply(lambda x: clean_text(x, True))

In [ ]:
data["text_clean"]

### Frequency Analysis

Now we want to perform frequency analyis. Specifically, we want to find the most frequent words for each class. To do so, we need to tokenize the tweets first. We'll use Spacy.

In [ ]:
nlp = spacy.load("it_core_news_sm")

nlp = it_core_news_sm.load()

In [ ]:
def str_freq_analysis(data: DataFrame, column_name) -> Counter:
    """
    Performs frequency analysis on the text data in the specified column.

    This function tokenizes the text data using the spaCy NLP pipeline and counts the frequency
    of words, excluding punctuation and specific parts of speech such as determiners, adpositions,
    pronouns, auxiliaries, conjunctions, particles, and punctuation.

    Args:
        data (DataFrame): A pandas DataFrame containing the text data.
        column_name (str): The name of the column in the DataFrame containing the text to analyze.

    Returns:
        Counter: A Counter object containing the frequency of words in the specified text column.
    """
    freqs = Counter()

    for doc in nlp.pipe(data[column_name], batch_size=50):
        freqs.update(
            [
                token.text.lower()
                for token in doc
                if not token.is_punct
                and token.pos_
                not in {"DET", "ADP", "PRON", "AUX", "CCONJ", "SCONJ", "PART", "PUNCT"}
            ]
        )

    return freqs

In [ ]:
def list_freq_analysis(data: DataFrame, column_name: str) -> Counter:
    """
    Performs frequency analysis on a column containing lists of items.

    Args:
        data (DataFrame): A pandas DataFrame containing the data.
        column_name (str): The name of the column containing lists of items.

    Returns:
        Counter: A Counter object with the frequency of each item in the lists.
    """
    freqs = Counter()

    # Ensure the column exists
    if column_name not in data.columns:
        raise ValueError(f"Column '{column_name}' does not exist in the DataFrame.")

    # Iterate through the column and update the counter
    for items in data[column_name]:
        if isinstance(items, list):  # Ensure the value is a list
            freqs.update(items)

    return freqs

In [ ]:
def plot_common(ax, freqs: Counter, plot_title: str, n_common: int = 10) -> None:
    """
    Plots a horizontal bar chart to visualize the most common items or hashtags or emojis and their relative frequencies.

    Args:
        ax (matplotlib.axes.Axes): The axes object to plot the bar chart on.
        freqs (Counter): A Counter object containing item frequencies.
        n_common (int, optional): The number of most common items to display. Defaults to 10.

    Returns:
        None
    """
    common_items = freqs.most_common(n_common)

    items = []
    rel_frequencies = []
    total_items = freqs.total()

    for word, frequency in common_items:
        items.append(word)
        rel_frequencies.append(frequency / total_items)

    y_pos = np.arange(len(items))

    ax.barh(y_pos, rel_frequencies, color="skyblue")
    ax.set_yticks(y_pos, labels=items)
    ax.invert_yaxis()
    ax.set_xlabel("Relative frequency")
    ax.set_title(plot_title)

In [ ]:
data_reclamation = data[data["label"] == 1]
data_no_rec = data[data["label"] == 0]

In [ ]:
freqs_hashtags = list_freq_analysis(data, "hashtags")

print(freqs_hashtags.most_common(20))

In [ ]:
plt.rcParams["font.family"] = "Segoe UI Emoji"

fig, axs = plt.subplots(4, 2, figsize=(16, 20))

freqs_all = str_freq_analysis(data, "text_clean")
freqs_rec = str_freq_analysis(data_reclamation, "text_clean")
freqs_no_rec = str_freq_analysis(data_no_rec, "text_clean")
freqs_hashtags_rec = list_freq_analysis(data_reclamation, "hashtags")
freqs_hashtags_no_rec = list_freq_analysis(data_no_rec, "hashtags")
freqs_emojis_rec = list_freq_analysis(data_reclamation, "emojis")
freqs_emojis_no_rec = list_freq_analysis(data_no_rec, "emojis")

plot_common(axs[0, 0], freqs_all, "word frequency for all the tweets")
plot_common(axs[1, 0], freqs_rec, "word frequency for reclamation tweets")
plot_common(axs[1, 1], freqs_no_rec, "word frequency for non reclamation tweets")
plot_common(axs[2, 0], freqs_hashtags_rec, "hashtag frequency for reclamation tweets")
plot_common(
    axs[2, 1], freqs_hashtags_no_rec, "hashtag frequency for non reclamation tweets"
)
plot_common(axs[3, 0], freqs_emojis_rec, "emoji frequency for reclamation tweets")
plot_common(
    axs[3, 1], freqs_emojis_no_rec, "emoji frequency for non reclamation tweets"
)

plt.show()

The plots give us some insights:
- In the most frequent words of the non-reclamation tweets, there are no slurs. The words *gay* and *trans* can sometimes be used as slurs, but they usually have a neutral meaning.
- In the reclamation tweets, different variations of *frocio* (e.g., frocio, forcio, forci, froci, frocia) are among the most frequent words. Some of these are misspelled, which could pose challenges during the model's training phase.
- Words that suggest reclamation, such as *pride*, are not particularly informative in distinguishing the label of the tweet.
- Some emojis seem to be informative with respect to the tweet label.

### Characteristic Term Analysis

In [ ]:
# Import hurtlex data from TSV
hurtlex = pd.read_csv("../data/raw/hurtlex_IT.tsv", sep="\t")

hurtlex.head()

In [ ]:
homophobic_terms = hurtlex[hurtlex["category"] == "om"]

homophobic_terms

In [ ]:
def generate_inflected_forms(word: str) -> set[str]:
    """Generate masculine/feminine and plural forms for a given Italian word."""
    forms = {word}

    if re.search(r"o$", word):  # masculine singular
        forms.update([word[:-1] + "a", word[:-1] + "i", word[:-1] + "e"])
    elif re.search(r"a$", word):  # feminine singular
        forms.update([word[:-1] + "o", word[:-1] + "i", word[:-1] + "e"])
    elif re.search(r"e$", word):  # could be masculine or feminine
        forms.add(word[:-1] + "i")
    # else: invariable (ends with consonant, accent, etc.)

    return forms


def expand_lexicon(lexicon: list[str]) -> set[str]:
    """Expand all words in the lexicon with their inflected forms."""
    all_forms = set()
    for w in lexicon:
        all_forms.update(generate_inflected_forms(w.lower()))
    return all_forms

In [ ]:
lex_list = homophobic_terms["lemma"].to_list()
lexicon_set = expand_lexicon(lex_list)

In [ ]:
def lex_freq_analysis(
    data: DataFrame, column_name: str, lexicon: set[str], nlp: spacy.Language
) -> Counter:
    """
    Performs frequency analysis on the text data in the specified column.

    This function tokenizes the text data using the spaCy NLP pipeline and counts the frequency
    of words that are present in the provided lexicon, excluding punctuation and specific parts
    of speech such as determiners, adpositions, pronouns, auxiliaries, conjunctions, particles,
    and punctuation.

    Args:
        data (DataFrame): A pandas DataFrame containing the text data.
        column_name (str): The name of the column in the DataFrame containing the text to analyze.
        lexicon (set[str]): A set of words to filter and count in the text data.
        nlp: A spaCy NLP pipeline for tokenization and linguistic analysis.

    Returns:
        Counter: A Counter object containing the frequency of words in the specified text column
        that are present in the lexicon.
    """
    slur_counts = Counter()

    for doc in nlp.pipe(data[column_name], batch_size=50):
        slur_counts.update(
            [
                token.text.lower()
                for token in doc
                if token.text.lower() in lexicon
                and not token.is_punct
                and token.pos_
                not in {"DET", "ADP", "PRON", "AUX", "CCONJ", "SCONJ", "PART", "PUNCT"}
            ]
        )

    return slur_counts

In [ ]:
slur_counts_rec = lex_freq_analysis(data_reclamation, "text_clean", lexicon_set, nlp)
slur_counts_normal = lex_freq_analysis(data_no_rec, "text_clean", lexicon_set, nlp)

In [ ]:
print(slur_counts_normal)

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(16, 20))

plot_common(axs[0], slur_counts_rec, "slurs in reclamation tweets")
plot_common(axs[1], slur_counts_normal, "slurs in non reclamation tweets")

Only two terms seem to be informative: frocia and finocchio.